# PISA STUDENT CSV FILTER

In [4]:
import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1.  File paths and ingestion
# ------------------------------------------------------------------
DATAFRAME_PATH = "data/pisa_2022/p22_stu_qqq_swe_nec.csv"
CODEBOOK_PATH  = "data/pisa_2022/filtered_codebook.csv"

data_df     = pd.read_csv(os.path.join(os.getcwd(), DATAFRAME_PATH))
codebook_df = pd.read_csv(os.path.join(os.getcwd(), CODEBOOK_PATH))

print(f"Loaded {len(data_df)} rows and {len(data_df.columns)} columns from {DATAFRAME_PATH}")

# Keep only codebook rows that correspond to columns actually present
codebook_df = codebook_df[codebook_df["NAME"].isin(data_df.columns)]

# ------------------------------------------------------------------
# 2.  Column‑level filtering (USE == 0 ➜ drop)
# ------------------------------------------------------------------
cols_to_drop = codebook_df.loc[codebook_df["USE"] == 0, "NAME"]
data_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
codebook_df = codebook_df[codebook_df["USE"] != 0]      # sync

# ------------------------------------------------------------------
# 3.  Value cleansing, mode imputation, and numerical adjustment
# ------------------------------------------------------------------
for _, row in codebook_df.iterrows():
    col        = row["NAME"]
    min_val    = row["MIN"]
    max_val    = row["MAX"]
    adjust_val = row.get("ADJUST", 0) if not pd.isna(row.get("ADJUST", 0)) else 0

    # Guard against surprises: skip if column vanished earlier
    if col not in data_df.columns:
        continue

    # Coerce everything to numeric; non‑numbers become NaN automatically
    data_df[col] = pd.to_numeric(data_df[col], errors="coerce")

    # 3a. anything outside [min, max] ➜ NaN
    mask_outside = (data_df[col] < min_val) | (data_df[col] > max_val)
    data_df.loc[mask_outside, col] = np.nan

    # 3b. mode (ignoring NaNs).  If multiple modes, take the first.
    try:
        mode_val = data_df[col].mode(dropna=True).iloc[0]
    except IndexError:
        # Entire column is NaN – leave it as is
        mode_val = np.nan

    # 3c. fill NaNs with mode
    if not np.isnan(mode_val):
        data_df[col].fillna(mode_val, inplace=True)

    # 3d. apply ADJUST
    if adjust_val != 0:
        data_df[col] = data_df[col] + adjust_val

# ------------------------------------------------------------------
# 4.  One‑hot / dummy variables for CATEGORICAL columns
# ------------------------------------------------------------------
categorical_rows = codebook_df[codebook_df["CATEGORICAL"] == 1]

for _, row in categorical_rows.iterrows():
    col     = row["NAME"]
    min_val = int(row["MIN"])
    max_val = int(row["MAX"])

    # Skip if column disappeared
    if col not in data_df.columns:
        continue

    # Generate dummy columns for every value in [min, max]
    for val in range(min_val, max_val + 1):
        new_col = f"{col}_{val}"
        data_df[new_col] = (data_df[col] == val).astype(int)

    # Drop the original categorical column
    data_df.drop(columns=[col], inplace=True)

print(f"Transformed {len(data_df)} rows and {len(data_df.columns)} columns")

# ------------------------------------------------------------------
# 5.  The dataframe `data_df` is now fully transformed.
# ------------------------------------------------------------------
# Example of saving it (optional):
# 
RESULT_PATH = os.path.join(os.getcwd(), "data/pisa_2022/pisa_clean.csv")


os.makedirs(os.path.dirname(RESULT_PATH), exist_ok=True)
data_df.to_csv(RESULT_PATH, index=False)
print(f"Saved cleaned data to {RESULT_PATH}")


Loaded 6072 rows and 725 columns from data/pisa_2022/p22_stu_qqq_swe_nec.csv


C:\Users\jonat\AppData\Local\Temp\ipykernel_11304\114792928.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data_df[col].fillna(mode_val, inplace=True)
C:\Users\jonat\AppData\Local\Temp\ipykernel_11304\114792928.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example,

Transformed 6072 rows and 741 columns
Saved cleaned data to c:\Users\jonat\OneDrive\Documents\6.University\()Degree_Project\StudentPerformance_ML\data/pisa_2022/pisa_clean.csv
